# Lab 04 — EDA — TV, Radio, Newspaper vs Sales
**First Real EDA Track** · Beginner · ~45 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Load data with pandas and summarise with describe()
2. Compute and interpret a correlation matrix
3. Create scatter plots and quartile-binned means
4. Write a 5-bullet findings memo with design caveats

## Datasets (this folder)
- `Advertising.csv` — auto-download from `https://raw.githubusercontent.com/justmarkham/scikit-learn-videos/master/data/Advertising.csv`

## How to run on Google Colab
1. Click **Start Lab** — or open the hosted notebook directly: [Open in Colab](https://colab.research.google.com/github/matheshcp/ai_course_content/blob/main/course-01-foundations-python-math-data/labs/lab-04-eda-advertising/lab-04-eda-advertising.ipynb) — it opens under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Run **Cell 0** first — it downloads `dataset.zip` with wget, unzips it, and every code cell below reads those unzipped files.
3. **Runtime → Run all** (GPU not required for Course 1).
4. Work the **Exercises** cells before revealing **Solutions**.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 (`wget dataset.zip` + `unzip`) → `Runtime → Run all`.


### Setup (dataset)

Run the next cell (Cell 0) once: it downloads `dataset.zip` with wget and unzips it next to the notebook. All code below reads these unzipped files (`Advertising.csv`). Skips the download when the files already exist.


In [ ]:
# Cell 0 — dataset first: wget dataset.zip + unzip (run this cell first).
import os, shutil, subprocess, urllib.request, zipfile

LAB_ID = "lab-04-eda-advertising"
DATASET_ZIP_URL = "https://raw.githubusercontent.com/matheshcp/ai_course_content/main/course-01-foundations-python-math-data/labs/lab-04-eda-advertising/bundle/dataset.zip"
NEED = ["Advertising.csv"]  # unzipped files used by the code below

def _have_files():
    return all(os.path.exists(f) for f in NEED)

def _wget_zip(url, dest):
    # shell equivalent: !wget -q <url> -O dataset.zip
    if shutil.which("wget"):
        subprocess.run(["wget", "-q", url, "-O", dest], check=True)
    else:  # plain Python without wget: stdlib fallback
        urllib.request.urlretrieve(url, dest)

if _have_files():
    print("dataset ready:", ", ".join(NEED))
else:
    _wget_zip(DATASET_ZIP_URL, "dataset.zip")
    # shell equivalent: !unzip -o -q dataset.zip
    with zipfile.ZipFile("dataset.zip") as z:
        z.extractall(".")
    print("downloaded + unzipped dataset.zip ->", ", ".join(NEED))


## First Real EDA Track: Correlation, Plots, Findings Memo

> **Scenario:** Marketing asks *which channel drives Sales*. Load `Advertising.csv` (200 rows), compute correlations, plot scatters, bin TV into quartiles, and write a 5-bullet findings memo.
>
> **You will learn:** pandas `read_csv` / `describe` / `corr`, matplotlib scatter, correlation vs causation.
> **Time:** ~45 minutes. **Level:** Beginner. **Needs:** pandas + matplotlib. **Env:** 🟢 Colab only.

Before any code, fix the decision in your head: marketing will not fund TV, Radio, and Newspaper equally forever, and your job this session is to say which channel looks most productive, show the evidence, and be explicit about what this dataset *cannot* prove. This is an observational EDA — no randomization, one market, n = 200 — so every number you print is an *association*, not a causal effect. The lab's decision question therefore has three parts: how big and spread is the data, what moves with Sales, and how confidently can you write that up.

### EDA mental map

| Question | Statistic / view | Code |
|---|---|---|
| How big / spread? | `describe()` | `df.describe()` |
| What moves with Sales? | Pearson r | `df.corr()["Sales"]` |
| Linear relationship? | scatter | `plt.scatter(df["TV"], df["Sales"])` |
| Dose–response? | binned means | `pd.qcut` + `groupby.mean` |

Read the table left to right. Each row is a question a stakeholder could ask out loud; the middle column is the statistical tool that answers it; the right column is the one-liner you will actually type. `describe()` establishes shape, center, and outliers before you trust anything else. `corr()` ranks linear association with Sales so you know where to look first. The scatter panels then check that a correlation is a real cloud of points, not two outliers dragging a line. Binned means finish the story by testing a dose–response trend: does Sales rise steadily as TV spend rises, quartile by quartile? The five numbered sections below execute this map in order and end with a written memo — the actual deliverable.

---

### 1. Load data (local first, Colab fallback)

**Why:** Nothing downstream is trustworthy until you know the table's shape and each column's range. This step also sets the lab's loading pattern — use a local `Advertising.csv` when present, otherwise download it — so later labs reuse the same habit. A one-line `describe()` catches unit surprises (dollars vs thousands) before they poison every chart you draw after this.

In [ ]:
import os
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # headless-safe; drop in interactive notebooks
import matplotlib.pyplot as plt

def load_ads():
    local = "Advertising.csv"
    if not os.path.exists(local):
        import urllib.request
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/justmarkham/scikit-learn-videos/master/data/Advertising.csv",
            local,
        )
    df = pd.read_csv(local)
    # First column is an unnamed index
    if str(df.columns[0]).startswith("Unnamed"):
        df = df.rename(columns={df.columns[0]: "index"})
    return df

df = load_ads()
print(df.shape)   # (200, 5)
print(df.head(3))
print(df.describe().round(2))


**What to notice:**
- `df.shape` is `(200, 5)` — an unnamed index column plus `TV`, `Radio`, `Newspaper`, `Sales`.
- `Sales` mean ≈ **14.02**, min **1.6**, max **27.0** — that spread is the variation you are trying to explain.
- `TV` mean ≈ **147.04** with a wide range, so raw spend levels across channels are not comparable without normalization.
- The three channel columns are all positive spend amounts; there are no obvious categorical or missing-value columns to clean first.

> **Pitfall:** `describe()` summarizes *this* sample only. With n = 200 in a single market, means and quartiles will move if you add another week of data — always quote n and the design (observational, one market) next to any summary number you paste into the memo.

Expected highlights: `Sales` mean ≈ 14.02, min 1.6, max 27.0; `TV` mean ≈ 147.04.

---

### 2. Correlation matrix

**Why:** The decision question is "which channel is most associated with Sales," and Pearson's r is the fastest way to rank the three candidates on a single scale (−1 to +1). Computing the full matrix at once also reveals *feature–feature* relationships — if two channels move together, their individual correlations with Sales will double-count the same signal. That single table therefore drives both your headline ranking and your caveats.

In [ ]:
corr = df[["TV", "Radio", "Newspaper", "Sales"]].corr().round(4)
print(corr)


Expected `Sales` column:

| Feature | r with Sales |
|---|---|
| TV | **0.7822** |
| Radio | 0.5762 |
| Newspaper | 0.2283 |

**What to notice:**
- TV is the clear leader at r ≈ **0.7822** — a strong positive linear association with Sales.
- Radio is moderate at r ≈ **0.5762**; Newspaper is weak at r ≈ **0.2283**.
- Off-diagonal feature pairs matter too: Radio–Newspaper ≈ **0.35**, while both TV pairs are near **0.05** (TV spend is essentially independent of the other two channels in this sample).
- Correlations are symmetric and diagonal-perfect (r = 1), so only the off-diagonal numbers carry information.

In [ ]:
print("strongest single-feature |r| with Sales:",
      corr["Sales"].drop("Sales").abs().idxmax())  # TV


> **Pitfall:** Correlation ≠ causation: this is observational spend data, not a randomized test. A high r only says the two columns move together in these 200 rows — confounders (seasonality, overall budget growth, market size) could drive both. Say “associated with”, not “drives”, in the memo; the design limit belongs in the same bullet as the number.

---

### 3. Scatter matrix (three panels)

**Why:** A single r can hide almost anything: curvature, heteroscedasticity, or two outlier points dragging the line. Side-by-side scatters let you *see* the association you just ranked, channel by channel, with the r printed on each panel as a caption. If the cloud looks linear and the title matches the matrix, your correlation summary is safe to report; if not, you must qualify it.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
for ax, col in zip(axes, ["TV", "Radio", "Newspaper"]):
    ax.scatter(df[col], df["Sales"], alpha=0.6, edgecolors="none")
    ax.set_xlabel(col)
    ax.set_ylabel("Sales")
    ax.set_title(f"{col} vs Sales  r={df[col].corr(df['Sales']):.3f}")
fig.tight_layout()
fig.savefig("eda_scatters.png", dpi=120)
print("saved eda_scatters.png")


**What to notice:**
- The **TV** panel shows a clear upward band — the strongest and most linear-looking relationship (title r ≈ 0.782).
- The **Radio** panel also trends up but with more vertical scatter (title r ≈ 0.576).
- The **Newspaper** panel is a diffuse cloud with only a faint tilt (title r ≈ 0.228) — weak linear signal, easy to over-read.
- `sharey=True` keeps the Sales axis identical across panels, so panel heights are directly comparable.
- The figure is written to `eda_scatters.png` for inclusion in the memo or report.

> **Pitfall:** Do not infer causation from a pretty upward cloud. The TV panel shows association across whatever markets and weeks produced these rows; only a randomized geo test (or a credible natural experiment) would justify the word “drives.”

---

### 4. TV quartiles → mean Sales

**Why:** Pearson r compresses the whole relationship into one number, which can be inflated by a handful of extreme points. Splitting TV into four equal-count bins and taking mean Sales per bin checks for a *dose–response* pattern: if each higher quartile has a higher mean, the association is broad-based, not an outlier trick. That monotone pattern is much easier to defend in a memo than a bare r.

In [ ]:
df["TV_bin"] = pd.qcut(df["TV"], 4, labels=["Q1", "Q2", "Q3", "Q4"])
tv_bins = df.groupby("TV_bin", observed=True)["Sales"].agg(["mean", "count"]).round(3)
print(tv_bins)


Expected (edges ≈ 74.38 / 149.75 / 218.83):

| TV_bin | mean Sales | n |
|---|---|---|
| Q1 | 8.390 | 50 |
| Q2 | 12.686 | 50 |
| Q3 | 16.548 | 50 |
| Q4 | 18.466 | 50 |

**What to notice:**
- Each bin holds exactly **50** rows — `pd.qcut(..., 4)` splits on quantiles, not fixed width, so group sizes are balanced by construction.
- Mean Sales rises monotonically: **8.390 → 12.686 → 16.548 → 18.466** — a clean dose–response with TV spend.
- The Q4 − Q1 gap is about **10.08** Sales units, a large practical lift on a 0–27 scale.
- Monotone increase with TV spend — stronger evidence than raw r alone, because no single quartile is carrying the whole trend.

In [ ]:
ax = tv_bins["mean"].plot(kind="bar", figsize=(6, 3), rot=0,
                          title="Mean Sales by TV quartile")
ax.set_ylabel("Mean Sales")
fig = ax.get_figure(); fig.tight_layout(); fig.savefig("eda_tv_quartiles.png", dpi=120)


**What to notice (chart):**
- Four ascending bars make the quartile trend readable at a glance — this is the figure to paste next to bullet 2 of the memo.
- The output file `eda_tv_quartiles.png` is what you would attach to a slide or notebook markdown cell.

> **Pitfall:** Binning throws away within-bin detail and the cut points are sample-dependent. Quartile means are a *robustness check* on the correlation, not a replacement for it — report both, and remember the underlying data are still observational.

---

### 5. Five-bullet findings memo

**Why:** EDA that stops at charts has not answered the decision question. The memo forces you to compress every finding above into claims a marketing lead can act on, each tied to a number, plus an explicit design caveat so nobody over-interprets the ranking. Writing it as five bullets also makes gaps obvious — if you cannot state a number, you have not finished the analysis.

Write `labs/eda_findings.md` (or cell markdown) with bullets like:

1. **TV is the strongest linear associate of Sales** (r ≈ 0.78), ahead of Radio (0.58) and Newspaper (0.23).
2. **Mean Sales rises monotonically across TV quartiles** (8.4 → 18.5), so the relationship is not driven by a few outliers alone.
3. **Newspaper’s weak r (0.23) largely tracks Radio** (r(Newspaper, Radio) ≈ 0.35); partial correlation / regression would likely shrink Newspaper’s unique contribution.
4. **TV and Radio spend are nearly uncorrelated** (r ≈ 0.05) — channels can be evaluated independently in this sample.
5. **Design limit:** observational data, n = 200, one market — treat findings as hypotheses for a geo test, not proof of causality.

In [ ]:
memo = """# EDA findings — Advertising channels
- TV strongest correlate of Sales (r=0.78).
- Mean Sales rises monotonically by TV quartile (8.4→18.5).
- Newspaper weak alone (r=0.23); overlaps Radio.
- TV vs Radio spend ~uncorrelated (r=0.05).
- Observational, n=200: associations only, not causal proof.
"""
open("eda_findings.md", "w", encoding="utf-8").write(memo)
print("wrote eda_findings.md")


**What to notice:**
- Every bullet cites a number you already computed — r values from step 2, quartile means from step 4 — so nothing in the memo is unsourced.
- Bullet 5 is not optional: it is the sentence that keeps the ranking honest given the observational design.
- The written file `eda_findings.md` is the lab's deliverable alongside the two PNG figures.

> **Pitfall:** A memo that only ranks channels invites “TV causes sales” reading. Keep the association/causation wording and the n = 200 caveat in the same document as the ranking, or the caveat will be dropped in the slide deck.

---

## Exercises (do these!)

### Exercise 1 — Full correlation matrix
Print `df[["TV","Radio","Newspaper","Sales"]].corr()` rounded to 4 d.p. Which pair of *features* (not Sales) is most correlated?
*Expected: Radio–Newspaper ≈ 0.3541 is the strongest feature–feature pair (TV pairs are ≈ 0.05).*

**Follow-up:** What fraction of Sales variance does TV alone explain (R squared)? Check: about 0.6118.

<details>
<summary>Hint</summary>

```python
c = df[["TV", "Radio", "Newspaper"]].corr()
print(c)
# upper triangle only to avoid duplicates
```

</details>

### Exercise 2 — Highest single-feature R vs Sales
Using absolute correlation with `Sales`, which feature wins and what is r?
*Expected: TV, r ≈ 0.7822.*

**Follow-up:** How large is the Q4 minus Q1 mean-sales lift? Check: 10.076.

<details>
<summary>Hint</summary>

`df.corr()["Sales"].drop("Sales").abs().idxmax()` / `.max()`.
</details>

### Exercise 3 — TV quartiles → mean Sales
Bin `TV` into 4 equal-count bins (`pd.qcut(..., 4)`), print mean `Sales` per bin.
*Expected: Q1 8.39 · Q2 12.686 · Q3 16.548 · Q4 18.466 (50 rows each).*

**Follow-up:** How much variance do Radio and Newspaper share (r squared)? Check: about 0.1254.

<details>
<summary>Hint</summary>

`pd.qcut(df["TV"], 4, labels=["Q1","Q2","Q3","Q4"])` then `groupby(...).mean()`.
</details>

---

## Solutions

In [ ]:
# --- Solution 1 ---
print(df[["TV", "Radio", "Newspaper", "Sales"]].corr().round(4))
# Radio-Newspaper ≈ 0.3541 is the strongest non-Sales pair.

# --- Solution 2 ---
s = df.corr(numeric_only=True)["Sales"].drop("Sales")
top = s.abs().idxmax()
print(top, round(s[top], 4))  # TV 0.7822

# --- Solution 3 ---
df["TV_bin"] = pd.qcut(df["TV"], 4, labels=["Q1", "Q2", "Q3", "Q4"])
print(df.groupby("TV_bin", observed=True)["Sales"].mean().round(3))
# Q1  8.390
# Q2 12.686
# Q3 16.548
# Q4 18.466

# --- Follow-up 1 ---
r = round(float(corr.loc["TV", "Sales"]), 4)
r2 = round(r ** 2, 4)
print(r2)  # ~0.6118
assert r2 == 0.6118

# --- Follow-up 2 ---
lift = round(tv_bins["mean"]["Q4"] - tv_bins["mean"]["Q1"], 3)
print(lift)  # 10.076
assert lift == 10.076

# --- Follow-up 3 ---
r_rn = round(float(corr.loc["Radio", "Newspaper"]), 4)
shared = round(r_rn ** 2, 4)
print(shared)  # ~0.1254
assert shared == 0.1254


### What to learn next
- Simple OLS: `sm.OLS(Sales, add_constant(df[["TV","Radio"]])).fit()` — does Newspaper add anything?
- Seaborn `pairplot(df, hue=None)` for a one-line scatter matrix.
- Then train/test split before trusting fit metrics (Course 2).
- Cheat sheet: describe → corr → scatter → binned means → memo; always state n and design limits.

*Files in this folder: `Advertising.csv` · outputs `eda_scatters.png`, `eda_tv_quartiles.png`, `eda_findings.md` when you run the lab.*

---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
